# Interactive Flight Planning Widgets

This notebook demonstrates the two interactive Jupyter widgets in
`hyplan.gui`:

1. **WaypointEditor** — place, drag, and edit waypoints on a map
2. **FlightLineManager** — add, select/deselect, and reorder flight lines

Both widgets share a `PlannerState` object and can operate on the same
ipyleaflet `Map`.

## Requirements

```bash
pip install hyplan[gui]
```

In [1]:
import hyplan
import ipyleaflet
from hyplan.gui import PlannerState, WaypointEditor, FlightLineManager
from hyplan.units import ureg

## 1. WaypointEditor

Click **Add Waypoint**, then click on the map to place waypoints.
Drag markers to reposition. Edit the table to change properties.

In [2]:
state1 = PlannerState()
m1 = ipyleaflet.Map(center=[34.9, -117.9], zoom=10)
editor = WaypointEditor(state1, m1, default_altitude=20000 * ureg.feet)
display(m1, editor)

Map(center=[34.9, -117.9], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…

WaypointEditor(children=(HBox(children=(ToggleButton(value=False, description='Add Waypoint', icon='plus', too…

In [3]:
# After placing waypoints, inspect them programmatically
for wp in state1.waypoints:
    print(f"{wp.name}: ({wp.latitude:.4f}, {wp.longitude:.4f})  hdg={wp.heading:.0f}")

## 2. FlightLineManager with pre-generated lines

Generate flight lines with `box_around_center_line()`, then use the
manager to select and reorder them for a flight plan.

In [4]:
from hyplan import box_around_center_line, AVIRISClassic

aviris = AVIRISClassic()
lines = box_around_center_line(
    instrument=aviris,
    altitude_msl=20000 * ureg.feet,
    lat0=34.9, lon0=-117.9,
    azimuth=0.0,
    box_length=30 * ureg.km,
    box_width=15 * ureg.km,
)
print(f"Generated {len(lines)} flight lines")

Generated 6 flight lines


In [5]:
state2 = PlannerState()
m2 = ipyleaflet.Map(center=[34.9, -117.9], zoom=10)
manager = FlightLineManager(state2, m2, flight_lines=lines)
display(m2, manager)

Map(center=[34.9, -117.9], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…

FlightLineManager(children=(HBox(children=(ToggleButton(value=False, description='Add Line', icon='plus', tool…

In [6]:
# After selecting and reordering, get the flight lines in order
selected = manager.selected_lines
print(f"{len(selected)} lines selected:")
for fl in selected:
    print(f"  {fl.site_name}: {fl.length.to('km'):.1f}")

6 lines selected:
  Line_L01_FL200: 30.0 kilometer
  Line_L02_FL200: 30.0 kilometer
  Line_L03_FL200: 30.0 kilometer
  Line_L04_FL200: 30.0 kilometer
  Line_L05_FL200: 30.0 kilometer
  Line_L06_FL200: 30.0 kilometer


## 3. Combined: both widgets sharing state + map

Use a single `PlannerState` and `Map` to work with both waypoints
and flight lines simultaneously.

In [7]:
state3 = PlannerState()
m3 = ipyleaflet.Map(center=[34.9, -117.9], zoom=10)

editor3 = WaypointEditor(state3, m3, default_altitude=20000 * ureg.feet)
manager3 = FlightLineManager(state3, m3, flight_lines=lines)

display(m3, editor3, manager3)

Map(center=[34.9, -117.9], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…

WaypointEditor(children=(HBox(children=(ToggleButton(value=False, description='Add Waypoint', icon='plus', too…

FlightLineManager(children=(HBox(children=(ToggleButton(value=False, description='Add Line', icon='plus', tool…

## 4. Compute a flight plan from selected lines

Pass the selected lines to `compute_flight_plan()` to generate
a complete flight plan with transit segments.

In [8]:
from hyplan import compute_flight_plan, DynamicAviation_B200, Airport

b200 = DynamicAviation_B200()
kedw = Airport("KEDW")  # Edwards AFB

selected = manager.selected_lines  # from Section 2
if selected:
    plan = compute_flight_plan(b200, selected, kedw, kedw)
    print(f"Flight plan: {len(plan)} segments")
    print(f"Total distance: {plan['distance'].sum():.1f} NM")
    print(f"Total time: {plan['time_to_segment'].sum():.1f} min")
    display(plan[["segment_type", "segment_name", "distance", "time_to_segment"]].head(10))
else:
    print("No lines selected — select some lines in Section 2 first.")

Flight plan: 13 segments
Total distance: 243.7 NM
Total time: 67.6 min


,segment_type,segment_name,distance,time_to_segment
0,takeoff,Departure,32.986306,8.909485
1,flight_line,Line_L01_FL200,22.198661,5.995780
2,transit,Line_L01_FL200 to Line_L02_FL200,8.040119,2.171608
3,flight_line,Line_L02_FL200,22.198652,5.995777
4,transit,Line_L02_FL200 to Line_L03_FL200,7.993628,2.159051
5,flight_line,Line_L03_FL200,22.198654,5.995778
6,transit,Line_L03_FL200 to Line_L04_FL200,8.038437,2.171153
7,flight_line,Line_L04_FL200,22.198709,5.995793
8,transit,Line_L04_FL200 to Line_L05_FL200,7.995131,2.159456
9,flight_line,Line_L05_FL200,22.198656,5.995779
